# Levels 2–4 on Colab: controls, methods, decider

Runs the benchmark scripts end to end and keeps results on Google Drive so a disconnect costs nothing (every run is append-only and resumable).

Budget on a T4 (≈1.3 s/step for SDXL at 1024²... measured, not assumed; your numbers go in `results.csv`):

| stage | images | rough T4 time |
|---|---|---|
| Level 2 controls (pilot) | 60 prompts × 2 seeds × 5 settings = 600 | ~5 h |
| Level 3 sweep (pilot) | 60 × 2 × 8 settings = 960 | ~6 h |
| Scoring | all | ~30 min |

Run a `--limit 6` smoke test first; then let each stage run in chunks (`--groups`) across sessions.

In [1]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/image-rerouter-git"  # code from GitHub; results persist here between sessions
    REPO_URL = "https://github.com/aaronh-e/image-rerouter"   # empty = upload a zip of the code instead
    OLD = "/content/drive/MyDrive/image-rerouter"       # the earlier zip-upload folder, if any
    if not os.path.exists(WORK):
        if REPO_URL:
            !git clone -q $REPO_URL $WORK
            if os.path.exists(f"{OLD}/results"):   # bring earlier results along once (the old folder is left as is)
                print(f"Copying results from {OLD} ...")
                !cp -r "$OLD/results" "$WORK/"
        else:
            from google.colab import files
            import glob, shutil
            print("Pick the image-rerouter zip (any name). This only happens once; after that it lives on your Drive.")
            ZIP = next(iter(files.upload()))
            !rm -rf /content/_unzip && unzip -q -o "$ZIP" -d /content/_unzip && rm "$ZIP"
            src = next(os.path.dirname(c) for c in glob.glob("/content/_unzip/**/config.yaml", recursive=True)
                       if os.path.isdir(os.path.join(os.path.dirname(c), "rerouter")))
            shutil.move(src, WORK)
    %cd $WORK
    if REPO_URL:   # a ! command must start its own line, or Colab reports invalid syntax
        !git pull -q
    else:
        print(f"Using the code already in {WORK} (your first upload). To update it, set REPO_URL, or delete that folder and re-run.")
    !pip install -q -r requirements.txt
    !pip uninstall -q -y torchao   # Colab's preinstalled torchao is too old for new diffusers (FqnToConfig import error); unused here
import torch; print(torch.cuda.get_device_name(0))

Mounted at /content/drive
/content/drive/MyDrive/image-rerouter-git
NVIDIA RTX PRO 6000 Blackwell Server Edition


In [2]:
# Dollar columns: $ per hour for the GPU this runtime got. Set USD_PER_HOUR to your price for it
# (Colab bills compute units: units per hour x $ per unit). None = use config.yaml's table, which
# only lists a T4; any other GPU then gets blank dollar columns instead of a T4's price.
USD_PER_HOUR = None
import os, torch
from rerouter import utils
if USD_PER_HOUR:
    os.environ["RR_USD_PER_HOUR"] = str(USD_PER_HOUR)   # bench/run.py reads it; no need to edit config.yaml
GPU = torch.cuda.get_device_name(0)
price = utils.usd_per_hour(utils.load_config(), GPU)
print("GPU:", GPU, "| price for $/1000 images:", f"${price}/h" if price else "none: dollar column stays blank (set USD_PER_HOUR above)")
cfg_steps = utils.load_config()["compute"]["step_seconds"]
print("% saved uses", "this GPU's measured step times" if GPU in cfg_steps else "step times fitted from this run's controls (plot.py prints them)")
# Hugging Face token (optional but avoids throttled downloads): Colab > Secrets > HF_TOKEN
try:
    from google.colab import userdata; os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception: pass

GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition | price for $/1000 images: none: dollar column stays blank (set USD_PER_HOUR above)
% saved uses this GPU's measured step times


## Smoke test (3 prompts, everything end to end, ~10 min)

In [ ]:
!python bench/run.py --controls --prompts prompts/pilot.csv --limit 3 --out results/smoke.csv --images results/smoke_images
!python bench/score.py --results results/smoke.csv --only vqa pick_score
!python bench/plot.py --results results/smoke.csv --out results/smoke_plots
from IPython.display import Image, display; display(Image("results/smoke_plots/tradeoff_vqa_score.png"))

## Level 2: controls
Large@30 (ceiling), small@30 (floor), large@12/18/24 (the line to beat). Re-run the same cell after any disconnect; finished rows are skipped.

In [ ]:
!python bench/run.py --controls --prompts prompts/pilot.csv

In [ ]:
!python bench/score.py --prompts prompts/pilot.csv
!python bench/plot.py --metric vqa_score
!python bench/plot.py --metric pick_score
from IPython.display import Image, Markdown, display   # each cell imports what it uses, so skipping a cell never breaks it
display(Image("results/plots/tradeoff_vqa_score.png"))
display(Image("results/plots/tradeoff_vqa_score_by_group.png"))
display(Markdown(open("results/plots/advantage_vqa_score.md").read()))

### Spot check: 10 random images against their scores (the plan's human sanity check)

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from PIL import Image as PILImage
df = pd.read_csv("results/results.csv")
df = df[~df.notes.astype(str).str.startswith("error")].sample(10, random_state=0)
fig, axes = plt.subplots(2, 5, figsize=(20, 9))
for ax, (_, r) in zip(axes.flat, df.iterrows()):
    ax.imshow(PILImage.open(r.image_path)); ax.axis("off")
    ax.set_title(f"{r.setting if 'setting' in r else r.direction}@{r.steps} VQA={r.vqa_score:.2f} PS={r.pick_score:.2f}\n{r.prompt[:40]}", fontsize=8)
plt.tight_layout(); plt.show()
# Write what you see vs. the scores in results/spot_check.md

## Level 3: the two methods
Both orders at switch fractions 0.2/0.4/0.6/0.8 (0 and 1 are the controls). Then the prediction checks:
- Method 1 (small first) should hurt **composition** → `count_match` on the composition group.
- Method 2 (large first) should hurt **text/hands** → `ocr_match` on the text group; hands by eye.

In [ ]:
!python bench/run.py --sweep --prompts prompts/pilot.csv

In [ ]:
!python bench/score.py --prompts prompts/pilot.csv
!python bench/plot.py --metric vqa_score
!python bench/plot.py --metric pick_score
!python bench/plot.py --metric count_match --groups composition
!python bench/plot.py --metric ocr_match --groups text
import pandas as pd
from IPython.display import Image, Markdown, display
display(Image("results/plots/tradeoff_vqa_score.png"))
display(Image("results/plots/tradeoff_vqa_score_by_group.png"))
display(Image("results/plots/heatmap_vqa_score.png"))
display(Markdown(open("results/plots/advantage_vqa_score.md").read()))
display(Markdown(open("results/plots/success_vqa_score.md").read()))
display(pd.read_csv("results/plots/order_head_to_head_vqa_score.csv"))

### Hands by eye
OCR and counting are automatic; hand quality is not. Show the hands_faces prompts at the same cost (s2l@0.4 vs l2s@0.6) side by side and tally which looks right.

In [3]:
import pandas as pd, matplotlib.pyplot as plt
from PIL import Image as PILImage
df = pd.read_csv("results/results.csv"); df = df[(df.group == "hands_faces") & (df.seed == 42)]
a = df[(df.direction == "s2l") & (df.switch_fraction.round(1) == 0.4)].set_index("prompt_id")
b = df[(df.direction == "l2s") & (df.switch_fraction.round(1) == 0.6)].set_index("prompt_id")
ids = sorted(set(a.index) & set(b.index))
fig, axes = plt.subplots(len(ids), 2, figsize=(8, 4 * len(ids)))
for row, pid in zip(axes, ids):
    row[0].imshow(PILImage.open(a.loc[pid].image_path)); row[0].set_title(f"{pid} s2l@0.4 (large finishes)", fontsize=8)
    row[1].imshow(PILImage.open(b.loc[pid].image_path)); row[1].set_title(f"{pid} l2s@0.6 (small finishes)", fontsize=8)
    for ax in row: ax.axis("off")
plt.tight_layout(); plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'results/results.csv'

## Level 4: the Jev decider
`decider_eval.py` builds the answer key from the sweep, splits prompts 70/30, asks Jev its typed questions, learns a profile→setting policy on the 70%, and scores the router on the 30% **by lookup** (every setting was already generated). The regex baseline runs first so you can see whether Jev adds anything.

In [ ]:
# Level 4 needs no GPU, so this cell also works on its own in a fresh runtime: it mounts Drive and
# moves into the repo folder from the setup cell (it must already hold results/results.csv from Level 3).
import os, sys
if "google.colab" in sys.modules and not os.path.exists("bench/decider_eval.py"):
    from google.colab import drive; drive.mount("/content/drive")
    %cd /content/drive/MyDrive/image-rerouter-git
    !pip install -q pandas pyyaml requests
try:
    from google.colab import userdata; os.environ["TYPESAFE_API_KEY"] = userdata.get("TYPESAFE_API_KEY")
except Exception: print("no TYPESAFE_API_KEY secret; only the heuristic baseline will run")
!python bench/decider_eval.py --prompts prompts/pilot.csv --provider heuristic
!cp results/router_eval.md results/router_eval_heuristic.md
if os.environ.get("TYPESAFE_API_KEY"):
    !python bench/decider_eval.py --prompts prompts/pilot.csv --provider typesafe
from IPython.display import Markdown, display
display(Markdown(open("results/router_eval.md").read()))

In [ ]:
# Optional: actually generate the routed images (adds Jev latency to wall clock, logged in notes) and put the router on the chart.
# !python bench/run.py --method router --policy results/decider_policy.json --prompts prompts/pilot.csv
# !python bench/score.py && python bench/plot.py

## Full set
Only after the pilot looks right. Same commands with `prompts/full.csv` (200 prompts): roughly 4× the time.

In [ ]:
# !python bench/run.py --controls --prompts prompts/full.csv
# !python bench/run.py --sweep --prompts prompts/full.csv
# !python bench/score.py --prompts prompts/full.csv && python bench/plot.py
# !python bench/decider_eval.py --prompts prompts/full.csv